## 01MIAR - Actividad Dataset

Raúl Contreras O.

## Introducción
 

El presente trabajo expone el uso de diferentes técnicas de programación en Python, ejecutadas en un entorno Jupyter Notebook, así como la aplicación de funciones internas de sus librerías para el tratamiento y análisis de datos.

En esta ocasión, utilizaremos las siguientes librerías esenciales:

   - NumPy: para el cálculo numérico y la manipulación eficiente de matrices.
   - Pandas: para la gestión y operaciones con datos estructurados.
   - Matplotlib: combinada con Pandas, nos permitirá la visualización de los datos, un proceso fundamental para comprender mejor la información.

A través de este enfoque visual y estructurado de la información, es posible desarrollar y explicar todo el proceso de limpieza, tratamiento y análisis de datos con mayor claridad y facilidad.
 

## Metodología

A continuazion se detalla la fuente escogida de  conjunto de datos para realizar el análisis y tratamiento de datos. Fuentes de datos:
OpenData del ayuntamiento de Valencia 

## Web API pública
- Interfaz de programación que se pone a disposición de cualquier desarrollador para acceder a datos o funcionalidades
### Valencia Open Data
- https://opendata.vlci.valencia.es/
#### Estaciones contaminación atmosférica
- https://opendata.vlci.valencia.es/dataset/estacions-contaminacio-atmosferiques-estaciones-contaminacion-atmosfericas
#### URL
URL: https://geoportal.valencia.es/server/rest/services/OPENDATA/MedioAmbiente/MapServer/156/query?where=1=1&outFields=*&f=json


## 1. Alcance del Trabajo y Justificación Técnica

### 1.1. Objetivo Principal del Análisis y Tratamiento
El propósito central de este proyecto consiste en diseñar e implementar un pipeline automatizado de consumo, limpieza y estructuración de datos crudos para transformarlos en activos de información óptimos para el análisis estadístico. 

El flujo de trabajo abarca desde el consumo de la respuesta estructurada en formato JSON emitida por la API del Geoportal, hasta su aplanamiento e integración en un objeto bidimensional (`DataFrame` de Pandas). Asimismo, se aplican técnicas vectorizadas de ingeniería de variables para realizar la extracción geométrica de las coordenadas espaciales ($X, Y$).

Para la fase de análisis visual y descriptivo, se ha seleccionado como métrica principal el **Dióxido de Nitrógeno (`no2`)**. Esta decisión responde a criterios estrictos de calidad del dato: la variable presenta un registro íntegro (100% libre de valores nulos o *NaN*).


### 1.3. Referencia de la Fuente de Datos
El estudio se nutre del conjunto de datos abiertos provisto por el **Ayuntamiento de Valencia** relativo a la red de estaciones de vigilancia de contaminación atmosférica. La captura de la información en tiempo real se realiza de forma directa a través del endpoint oficial del Geoportal institucional:

     -- Endpoint de Consulta de la API:
> (https://geoportal.valencia.es/server/rest/services/OPENDATA/MedioAmbiente/MapServer/156/query?where=1=1&outFields=*&f=json)
 


# Fase 1: Configuración Global e Definicion de Datos

En esta sección se definen los parámetros iniciales, las rutas del sistema de archivos y las dependencias necesarias para la ejecución del proyecto. Se establecen los endpoints oficiales del Geoportal de la ciudad de Valencia para garantizar la conexión con la API de datos abiertos de contaminación atmosférica.

In [1]:
# ==============================================================================================================================================
# 1. IMPORTACIÓN DE LIBRERÍAS 
# ==============================================================================================================================================
import os               # Interacción con el sistema operativo y creación de carpetas, incluye import path 
import requests         # Biblioteca estándar para realizar peticiones HTTP a APIs web
import json             # Módulo para procesar y formatear estructuras de datos JSON
import time             # Proporciona funciones relacionadas con el tiempo y el control de pausas (sleep)
import pandas as pd     # Herramienta fundamental para la manipulación y análisis de datos tabulares
import numpy as np      # Librería especializada en cálculo numérico y operaciones matriciales

# ==============================================================================================================================================
# 2. PARÁMETROS GLOBALES Y RUTAS DE ACCESO
# ==============================================================================================================================================
# Endpoint base del Geoportal del Ayuntamiento de Valencia para la calidad del aire
server_url = "https://geoportal.valencia.es/server/rest/services/OPENDATA/"
base_url = server_url + "MedioAmbiente/MapServer/156/query"

# Estructuración y resolución de la ruta del archivo CSV de forma compatible con cualquier sistema operativo
path_csv = ['res', 'valencia_pollution_dataset.csv']
path_csv_solved = os.path.join(*path_csv)

In [2]:
# ==============================================================================================================================================
# SUB-PASO 1.2: FUNCIÓN DE CONEXIÓN y CONSUMO DE API
# ==============================================================================================================================================
def make_request(base_url, params):
    """
    Realiza una petición HTTP GET a la API seleccionada para descargar datos en formato JSON.
    Input: base_url (str) -> Enlace del servidor; params (dict) -> Filtros de consulta.
    Output: dict -> El mapa de datos crudos descargado del servidor.
    """
    # 1. Enviar la solicitud de descarga al Geoportal de Valencia con los parámetros definidos
    response = requests.get(base_url, params=params)
    
    # 2. Control defensivo: verificar si el servidor respondió con éxito (Código 200 OK)
    if response:
        # Retornar el mapa de datos crudos convertido en un diccionario nativo de Python
        return response.json()
    else:
        # Si la conexión falla, forzar una excepción controlada para detener el script
        raise Exception("Error Descargando JSON: No se pudo conectar con el servidor.")

In [3]:
# ==============================================================================================================================================
# SUB-PASO 1.3: FUNCIÓN DE EXTRACCIÓN Y UNIÓN DE VARIABLES
# ==============================================================================================================================================
def Data_Transformation(raw_response):
    """
    Procesa la respuesta cruda de la API para desempaquetar y fusionar los datos estadísticos y geométricos.
    Input: raw_response (dict) -> El payload crudo del servidor.
    Output: list -> Una lista limpia de diccionarios, donde cada elemento representa una estación.
    """
    # 1. Acceder de manera directa al contenedor maestro 'features' que aloja las estaciones
    sub_list = raw_response['features']
    
    # 2. Bucle de Comprensión (List Comprehension) avanzado de doble cajón:
    # Une el diccionario de variables ('attributes') con el de coordenadas espaciales ('geometry')
    # utilizando el operador de unión de diccionarios (|).
    clean_rows = [station['attributes'] | station['geometry'] for station in sub_list]
    
    # 3. Retornar la lista homogénea y aplanada lista para su lectura en Pandas
    return clean_rows

### Sub-paso 1.4: Conversión de Estructuras a Pandas DataFrame

Una vez que los datos crudos del JSON han sido unificados en una lista homogénea de diccionarios, el siguiente paso consiste en estructurarlos en un objeto bidimensional. Para ello, utilizamos el constructor principal de la librería **Pandas**, transformando la lista en un DataFrame tabular (matriz de filas y columnas) que facilitará las fases de limpieza y análisis estadístico.

In [4]:
# ==============================================================================================================================================
# SUB-PASO 1.4: CREACIÓN DEL DATAFRAME TABULAR
# ==============================================================================================================================================
def build_dataframe(clean_data_list):
    """
    Transforma la lista de diccionarios unificados en un DataFrame estructurado de Pandas.
    Input: clean_data_list (list) -> Lista de diccionarios con datos y coordenadas.
    Output: df (DataFrame) -> Matriz tabular de Pandas lista para el análisis.
    """
    # 1. Convertir la lista de datos procesados en una estructura de tabla real de Pandas
    df = pd.DataFrame(clean_data_list)
    
    # 2. Línea opcional para visualización en el espacio de trabajo (desactivada por defecto)
    # display(df)
    
    # 3. CRUCIAL: Retornar la tabla estructurada fuera de la función para su almacenamiento
    return df

### Sub-paso 1.5: Persistencia de Datos en Almacenamiento Local (CSV)

Para asegurar que las capturas periódicas de información no se pierdan al cerrar el entorno de ejecución, se desarrolla un procedimiento de almacenamiento en disco. Esta función implementa una lógica de control de archivos adaptativa: si el archivo de almacenamiento no existe, se genera una estructura nueva incluyendo las cabeceras; si el archivo ya existe en el directorio, el sistema pasa de forma  automática a modo de anexado (`append`), asegurando que los nuevos registros se acoplen al final del documento sin duplicar la fila de títulos de las columnas.

In [5]:
# ==============================================================================================================================================
# SUB-PASO 1.5: ALMACENAMIENTO DINÁMICO EN DISCO (OPTIMIZADO)
# ==============================================================================================================================================
def save_csv(df_csv):
    """
    Guarda el DataFrame en el archivo CSV local gestionado en la configuración global,
    decidiendo dinámicamente si anexa registros o crea una estructura nueva.
    Input: df_csv (DataFrame) -> Tabla de Pandas con los nuevos datos capturados.
    """
    # Control adaptativo del sistema de archivos usando la variable global 'path_csv_solved'
    if os.path.exists(path_csv_solved):
        # Si el archivo ya existe: pasa a modo "Agregar" ('a') y omitir las cabeceras
        file_mode = 'a'
        has_header = False
    else:
        # Si es un archivo nuevo: pasa a modo escritura ('w') y escribir cabeceras  
        file_mode = 'w'
        has_header = True
        
    # Ejecución del método de exportación tabular utilizando las variables de entorno dinámicas
    df_csv.to_csv(
        path_csv_solved,      # Variable global heredada directamente de la celda de configuración
        sep=';',              # Separador estándar europeo para evitar conflictos con decimales
        header=has_header,    # Control dinámico de la fila de títulos
        index=False,          # Evitar escribir el índice secuencial numérico interno de Pandas
        mode=file_mode        # Control dinámico para el modo de apertura del archivo físico
    )

### Fase 4: Función de Ingeniería de Datos con NumPy

Para mantener la modularidad del Notebook, se encapsulan las operaciones matemáticas vectorizadas en una función de procesamiento. Este bloque utiliza la librería **NumPy** de manera explícita para resolver dos aspectos críticos de la salud del dataset: la imputación del Monóxido de Carbono (`co`) detectando valores nulos, y la categorización lógica de alertas de contaminación por Dióxido de Nitrógeno (`no2`) mediante condiciones booleanas.



In [6]:
# ==============================================================================================================================================
# FASE 4: FUNCIÓN DE ACTUALIZACIÓN MATRICIAL (NUMPY)
# ==============================================================================================================================================
def enrich_with_numpy(df_input):
    """
    Utiliza operaciones vectorizadas de NumPy para limpiar nulos en 'co'
    y segmentar niveles de alerta en 'no2'.
    Input: df_input (DataFrame) -> El DataFrame original con los datos crudos.
    Output: df_res (DataFrame) -> Una copia estructurada con las nuevas columnas calculadas.
    """
    # Hacer una copia para proteger el DataFrame original de modificaciones colaterales
    df_res = df_input.copy()
    
    # --- Operación 1: Tratamiento de Valores Faltantes (Columna CO) ---
    # Calculamos la media del Monóxido de Carbono omitiendo los registros vacíos
    media_co = df_res['co'].mean()
    # Si np.isnan detecta un nulo, implanta la media; si no, preserva el valor original de la celda
    df_res['co_limpio'] = np.where(np.isnan(df_res['co']), media_co, df_res['co'])
    
    # --- Operación 2: Clasificación de Alertas Críticas (Columna NO2) ---
    # Si el valor de NO2 supera los 40.0 µg/m³, se cataloga como zona de tráfico denso o alta contaminación
    df_res['estado_alerta'] = np.where(df_res['no2'] > 40.0, 'Alta Contaminación', 'Normal')
    
    # Retornar la nueva matriz de datos completamente enriquecida y saneada
    return df_res

## Fase 5: Pruebas Unitarias e Integración del Pipeline (Parte A)

En esta sección se ejecuta el flujo de trabajo de forma lineal y controlada. El objetivo es validar individualmente la respuesta del servidor del Geoportal, inspeccionar la estructura de los metadatos y asegurar la correcta transformación e integración de los componentes (Pandas, NumPy y Matplotlib) antes de proceder con la automatización cíclica.

In [7]:
# ==============================================================================================================================================
# TEST DE INTEGRACIÓN INDIVIDUAL (PARTE A: SIN CICLOS)
# ==============================================================================================================================================

# 1. Definición de los parámetros de consulta requeridos por el Geoportal de Valencia
query_params = {
    "where": "1=1",
    "outFields": "*",
    "f": "json"
}

print("✅====== PASO 1: INGESTA Y DIAGNÓSTICO ======")
# 2. Descarga de datos crudos desde el servidor mediante la función de peticiones
raw_data = make_request(base_url, params=query_params)
# Imprime los resultados
print("✅ Captura de datos crudos completada.")
print("✅ Diccionario de alto nivel: ")
print(raw_data.keys())
print("✅ Imprime toda la informacion sin procesar en formato JSON permite analizar la estructura: ->")
print(json.dumps(raw_data, indent=4))
print("  <--------✅")

# Acceder al contenedor de estaciones 'features' y contabilizar los registros activos
n_stations = len(raw_data['features'])
print(f"Número total de estaciones activas detectadas = {n_stations}")

# Inspeccionar las claves de alto nivel del diccionario devuelto por la API
print(f"Claves principales del payload JSON: {list(raw_data.keys())}")


print("\n✅====== PASO 2: TRANSFORMACIÓN E INGENIERÍA DE VARIABLES ======")
# 3. Extraer y aplanar los atributos y geometrías en una lista homogénea de diccionarios
clean_data_row = Data_Transformation(raw_data)
print("✅ Estructuras JSON aplanadas y mapeadas con éxito.")

# Mostrar el primer registro de estación de forma estética para validar la estructura interna
print("Estructura muestra de la primera estación procesada:")
print(json.dumps(clean_data_row[0], indent=4))

print("\n✅====== PASO 3: PIPELINE DE PANDAS Y NUMPY ======")
# 4. Construir el DataFrame de Pandas a partir de la lista saneada
my_pandas_table = build_dataframe(clean_data_row)

# ----------------------------------------------------------------------------------------------------------------------------------------------
# MUESTRA DE DATAFRAME ORIGINAL (PANDAS INICIAL)
# ----------------------------------------------------------------------------------------------------------------------------------------------
print("\n✅====== PASO 3.1: MUESTRA DE DATAFRAME ORIGINAL (PANDAS INICIAL)  ======")
# Mostramos las primeras filas de la tabla original antes de las operaciones matemáticas de NumPy.
# Aquí se puede observar cómo las columnas 'co' mantienen sus valores nulos (NaN) originales y los tipos de datos crudos.
print("📋 Tabla Estructurada Original (Pandas sin procesar):")
display(my_pandas_table[['nombre', 'co', 'no2']].head())

  
# 5. Aplicar la función matricial de NumPy para limpiar nulos de CO y categorizar alertas de NO2
df_enriquecido = enrich_with_numpy(my_pandas_table)
print(f"✅ Tabla de Pandas enriquecida con NumPy. Dimensiones finales: {df_enriquecido.shape}")

# ----------------------------------------------------------------------------------------------------------------------------------------------
# APLICACIÓN DE NUMPY Y COMPARATIVA DE TRANSFORMACIÓN TABULAR
# ----------------------------------------------------------------------------------------------------------------------------------------------
print("\n✅====== PASO 3.2: APLICACIÓN DE NUMPY Y COMPARATIVA DE TRANSFORMACIÓN TABULAR  ======")
# Ejecutamos el motor de ingeniería de variables de NumPy para inyectar la lógica de negocio
df_enriquecido = enrich_with_numpy(my_pandas_table)

# Mostramos la nueva tabla para evidenciar la transformación:
# 1. 'co_limpio' ha transformado los tipos faltantes sustituyéndolos por la media flotante calculada de forma vectorizada.
# 2. 'estado_alerta' genera una nueva variable categórica de tipo objeto (string) mediante evaluación condicional booleana.
print("\n📊 Tabla Saneada y Enriquecida (Transformación Vectorizada con NumPy):")
display(df_enriquecido[['nombre', 'co', 'co_limpio', 'no2', 'estado_alerta']].head())
 
print("\n====== PASO 4: PERSISTENCIA EN ALMACENAMIENTO LOCAL ======")
# 6. Almacenar el DataFrame final en el disco local gestionando el modo de escritura o anexado
save_csv(df_enriquecido)
print("✅ Archivo de almacenamiento 'res/valencia_pollution_dataset.csv' actualizado y seguro.")

✅====== PASO 1: INGESTA Y DIAGNÓSTICO ======
✅ Captura de datos crudos completada.
✅ Diccionario de alto nivel: 
dict_keys(['displayFieldName', 'fieldAliases', 'geometryType', 'spatialReference', 'fields', 'features'])
✅ Imprime toda la informacion sin procesar en formato JSON permite analizar la estructura: ->
{
    "displayFieldName": "nombre",
    "fieldAliases": {
        "objectid": "objectid",
        "nombre": "Nom / Nombre",
        "direccion": "Adre\u00e7a / Direccion",
        "tipozona": "Tipus Zona / Tipo Zona",
        "parametros": "Par\u00e0metres / Par\u00e1metros",
        "mediciones": "Mesuraments / Mediciones",
        "so2": "so2",
        "no2": "no2",
        "o3": "o3",
        "co": "co",
        "pm10": "pm10",
        "pm25": "pm25",
        "tipoemisio": "tipoemision",
        "fecha_carg": "fecha_carga",
        "calidad_am": "calidad_ambiental",
        "fiwareid": "fiwareid"
    },
    "geometryType": "esriGeometryPoint",
    "spatialReference": {
      

,nombre,co,no2
0,Centro,NaN,6.0
1,Dr. Lluch,NaN,10.0
2,Francia,0.0,6.0
3,Boulevar Sur,NaN,4.0
4,Pista de Silla,0.0,8.0


✅ Tabla de Pandas enriquecida con NumPy. Dimensiones finales: (11, 20)

✅====== PASO 3.2: APLICACIÓN DE NUMPY Y COMPARATIVA DE TRANSFORMACIÓN TABULAR  ======

📊 Tabla Saneada y Enriquecida (Transformación Vectorizada con NumPy):


,nombre,co,co_limpio,no2,estado_alerta
0,Centro,NaN,0.0,6.0,Normal
1,Dr. Lluch,NaN,0.0,10.0,Normal
2,Francia,0.0,0.0,6.0,Normal
3,Boulevar Sur,NaN,0.0,4.0,Normal
4,Pista de Silla,0.0,0.0,8.0,Normal



====== PASO 4: PERSISTENCIA EN ALMACENAMIENTO LOCAL ======
✅ Archivo de almacenamiento 'res/valencia_pollution_dataset.csv' actualizado y seguro.


In [8]:
# ==============================================================================================================================================
# FASE 5.2: REPRESENTACIÓN GRÁFICA DE CONTAMINANTES ATMOSFÉRICOS (MATPLOTLIB)
# ==============================================================================================================================================

# 1. Configurar las dimensiones iniciales del lienzo de dibujo (Ancho: 12, Alto: 6 pulgadas)
plt.figure(figsize=(12, 6))

# 2. Renderizar gráfico de barras asociando variables categóricas (Nombres) y numéricas (NO2)
plt.bar(
    df_enriquecido['nombre'], 
    df_enriquecido['no2'], 
    color='teal',          # Paleta de color corporativo (Verde azulado)
    edgecolor='black'      # Delineado perimetral para mejorar el contraste visual
)

# 3. Optimización de etiquetas del eje X: Rotación adaptativa para evitar solapamientos de texto
plt.xticks(rotation=45, ha='right')

# 4. Inyección de metadatos y rotulación científica del gráfico
plt.title("Nivel de Contaminación por Dióxido de Nitrógeno (NO2) por Estación", fontsize=14, fontweight='bold')
plt.xlabel("Estaciones de Monitoreo (Red de Vigilancia de Valencia)", fontsize=12)
plt.ylabel("Niveles de NO2 (µg/m³)", fontsize=12)

# 5. Ajuste automático de márgenes periféricos y renderizado final en pantalla
plt.tight_layout()
plt.show()

NameError: name 'plt' is not defined

## Fase 6: Análisis Estadístico Agrupado (Pandas GroupBy)

Para extraer conocimiento estratégico del dataset, aplicamos el operador `.groupby()` de **Pandas**. Este concepto hereda directamente la lógica de la cláusula `GROUP BY` utilizada en bases de datos relacionales (**SQL**), y sigue el principio metodológico conocido en ciencia de datos como **Split-Apply-Combine** (Dividir - Aplicar - Combinar):

1. **Split (Dividir):** El motor de Pandas segmenta las filas del DataFrame en "cajones" independientes basados en una variable categórica (en este caso, `tipozona`, dividiendo las estaciones en *Urbana* y *Suburbana*).
2. **Apply (Aplicar):** Se aísla la variable numérica de interés (`no2`) dentro de cada contenedor y se le aplica una función de agregación matemática (la media aritmética mediante `.mean()`, equivalente al `AVG()` de SQL).
3. **Combine (Combinar):** Los sub-totales calculados se vuelven a fusionar en una nueva estructura tabular compacta de alta densidad informativa.

### ¿Por qué realizamos este análisis?
Mirar las estaciones de forma individual nos da datos puntuales, pero no nos permite identificar patrones geográficos o de comportamiento. Agrupar los datos nos permite validar hipótesis científicas urbanas, como evaluar si la densidad del tráfico y la actividad comercial de las áreas puramente urbanas provocan un impacto directo en la calidad del aire en comparación con los entornos residenciales o suburbanos.

In [ ]:
# ==============================================================================================================================================
# DEMOSTRACIÓN DIDÁCTICA: CONTRASTE DE DATOS ANTES Y DESPUÉS DEL GROUPBY
# ==============================================================================================================================================

print("🔍 [VISTA 1] - DATOS CRUDOS POR ESTACIÓN (Tabla Extendida Original)")
print("Nota cómo los datos están dispersos y es difícil determinar una tendencia a simple vista:")
# Mostramos una selección de estaciones con sus zonas y niveles individuales de NO2
display(df_enriquecido[['nombre', 'tipozona', 'no2']].head(8))

print("\n" + "="*80 + "\n")

print("📊 [VISTA 2] - CONOCIMIENTO EXTRAÍDO (Tabla Compacta mediante GroupBy)")
print("Pandas ha segmentado, calculado las medias aritméticas y combinado el resultado:")

# Ejecutamos el flujo completo equivalente a SQL: SELECT tipozona, AVG(no2) FROM df_enriquecido GROUP BY tipozona;
tabla_resumen = df_enriquecido.groupby('tipozona')['no2'].mean()
display(tabla_resumen)  # Resumen del Group By

print("\n✍️  CONCLUSIÓN DEL ANÁLISIS:")
# Extracción automática de valores para la redacción de la defensa
media_suburbana = tabla_resumen.loc['Suburbana']
media_urbana = tabla_resumen.loc['Urbana']
diferencia_porcentaje = ((media_urbana - media_suburbana) / media_suburbana) * 100

print(f"Los datos demuestran que las estaciones situadas en áreas 'Urbanas' registran un promedio de NO2")
print(f"de {media_urbana:.2f} µg/m³, mientras que las 'Suburbanas' registran {media_suburbana:.2f} µg/m³.")
print(f"Esto confirma que el centro urbano tiene un {diferencia_porcentaje:.1f}% MÁS de contaminación por NO2,")
print("lo cual correlaciona directamente con la mayor densidad de tráfico rodado en la ciudad de Valencia.")

## Parte B: Automatización del Pipeline mediante Ciclos Temporales

En esta sección se implementa la simulación de un entorno de producción automatizado. Se utiliza un bucle condicional `while` combinado con el módulo `time` para generar capturas periódicas de la API del Geoportal. En cada iteración (intervalo de tiempo controlado), el script descarga de forma autónoma el estado de los sensores, aplica las transformaciones de limpieza, ejecuta el motor de enriquecimiento de **NumPy** y vuelca los resultados de forma incremental en el almacenamiento físico local sin interrumpir la ejecución del programa.

In [ ]:
# ==============================================================================================================================================
# PIPELINE AUTOMATIZADO CON TEMPORIZADOR (PARTE B)
# ==============================================================================================================================================

# 1. Configuración de parámetros de consulta y límites temporales
query_params = {
    "where": "1=1",
    "outFields": "*",
    "f": "json"
}

current_time = 0                          # Punto de partida del reloj interno (segundos)
sleep_time = 1                            # Tiempo de espera entre capturas (ajustado a 1s para pruebas rápidas)
total_time = 3                            # Duración total de la simulación (3 ciclos de ejecución)

print("🚀 Iniciando el sistema de monitorización automatizado...")

# 2. Bucle Principal: Controla el paso de las horas/ciclos del sistema
while current_time < total_time:
    print(f"\n⏱️ --- Marca de Reloj: {current_time} segundos ---")
    
    try:
        # A. Descarga en tiempo real: Se realiza DENTRO del ciclo para capturar actualizaciones reales
        raw_response = make_request(base_url, query_params)
        
        # B. Transformación: Aplanar las estructuras JSON extraídas del servidor
        clean_data_rows = Data_Transformation(raw_response)
        
        # C. Conversión a Estructura Tabular de Pandas
        df_ciclo_base = build_dataframe(clean_data_rows)
        
        # D. Ingeniería de Variables: Enriquecimiento explícito con NumPy (Nulos y Alertas)
        df_ciclo_enriquecido = enrich_with_numpy(df_ciclo_base)
        
        # E. Persistencia Adaptativa en Disco: Almacenamiento incremental seguro sin duplicar cabeceras
        save_csv(df_ciclo_enriquecido)
        
        # Diagnóstico por consola del ciclo actual
        registros_procesados = len(df_ciclo_enriquecido)
        print(f"   ✅ Datos almacenados con éxito. Registros capturados en este ciclo: {registros_procesados}")
        
    except Exception as e:
        # Control defensivo: Si un ciclo falla por red, el programa avisa pero continúa con el siguiente
        print(f"   ⚠️ Error detectado en este ciclo de captura: {e}")
    
    # F. Control del Tiempo: Incrementar el reloj y suspender el hilo del procesador
    current_time += sleep_time
    time.sleep(sleep_time)  # Pausa real controlada del sistema

print("\n🏁 Simulación automatizada finalizada. El archivo histórico local está actualizado.")

## Conclusión 

Este trabajo de investigación ha representado una excelente oportunidad para aplicar en un escenario real el concepto de listas de diccionarios, demostrando su relevancia estructural al trabajar con la ingesta de datos desde una API en formato JSON. Asimismo, el proyecto ha permitido dominar el flujo completo de la ciencia de datos: desde la captura y limpieza automatizada, pasando por la ingeniería de variables con NumPy para gestionar alertas y corregir valores nulos, hasta la estructuración en un DataFrame de Pandas. Finalmente, la integración con Matplotlib facilitó una visualización clara y efectiva de los niveles de contaminación atmosférica en Valencia, logrando transformar datos brutos en información analítica de gran valor.